In [ ]:
from langchain_community.document_loaders import PyPDFLoader


loader = PyPDFLoader("weed_dataset.pdf")
pages = loader.load()

print(f"Total pages: {len(pages)}")
 

In [ ]:
import re

for doc in pages:
    text = doc.page_content
    
    text = re.sub(r"\s+", " ", text)
    
    text = re.sub(r"\s*-\s*", "-", text)

    doc.page_content = text.strip()

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size =  800,
    chunk_overlap = 50,
)

chunks = text_splitter.split_documents(pages)
print(f"total chunks: {len(chunks)}")
print(chunks[0].page_content)

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from dotenv import load_dotenv

load_dotenv()

embeddings = OpenAIEmbeddings(model = "text-embedding-3-small")

vector_store = Chroma(
    collection_name = "weed_management",
    embedding_function = embeddings,
)

vector_store.add_documents(documents = chunks)

retriever = vector_store.as_retriever(search_kwargs={"k" : 5})


In [ ]:
from typing import TypedDict , List , Literal

class WeedState(TypedDict):
    original_question: str
    current_query : str
    gathered_facts : List[str]
    evaluation : str | None
    missing_info: str | None
    response: str
    chat_history : List

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o")

In [ ]:
from langchain_core.prompts import PromptTemplate

weed_prompt_template = """
You are an expert agricultural assistant helping 
farmers identify and manage noxious weeds.

Previous Conversation:
{chat_history}

Instructions:
- Only answer based on the verified facts provided
Detect the language of the user's question.
- If the question is in Tamil, answer in Tamil.
- If the question is in English, answer in English.
- If answer not in facts say 
  "I don't have information about that"
- For weed identification:
  * Mention exact species name
  * List matching characteristics
  * Differentiate from similar species
- For control methods:
  * Mention ALL methods: mechanical, chemical, biological
  * Include specific timing windows
  * Mention growth stages
- Be clear and simple for farmers

Verified Facts:
{facts}

Farmer's Question: {question}

Answer:"""

weed_prompt = PromptTemplate.from_template(weed_prompt_template)
weed_chain = weed_prompt | llm



In [ ]:
def retrieve_facts(state: WeedState):
    """Search the knowledge base for facts relevant to the query"""  
    
    query = state["current_query"]
    new_facts = state.get("gathered_facts", []).copy()
    docs = retriever.invoke(query)
    for doc in docs:
        if doc.page_content not in new_facts:
            new_facts.append(doc.page_content)
    print("\n\nFacts so far\n")
    print(*new_facts, sep="\n")
    return{"gathered_facts" : new_facts}

In [ ]:
from langchain_core.messages import SystemMessage , HumanMessage
from pydantic import BaseModel , Field
from typing import Literal

class Evaluation(BaseModel):
    status : Literal ["answerable", "needs_more", "irrelevant"] = Field(
        description = (
            "'answerable' - the facts fully answer the question"
            "'needs_more' - some relevant facts found , but a gap remains"
            "'irrelevant' - if question is not about weeds"
        )
    )
    reasoning : str = Field(description = "Brief explanation of what we know and what's missing")
    missing_piece : str = Field(default="" , description = "If needs_more: describe the specific information gap")

def evaluate(state: WeedState):
    question = state["original_question"]
    gathered = state.get("gathered_facts" , []) 
    facts_str = "\n".join(f"- {f}" for f in gathered ) if gathered else "(none)"

    eval_llm = llm.with_structured_output(Evaluation)
    result = eval_llm.invoke([
        SystemMessage(content=(
             "You are an expert evaluator for agricultural "
            "weed management.\n"
            "Given a farmer's question and gathered facts,\n"
            "determine if facts are sufficient to answer.\n"
            "- 'answerable': facts fully answer the question\n"
            "- 'needs_more': specific gap remains\n"
            "- 'irrelevant': the question is not related to weed identification, weed management, herbicides, invasive plants, or agriculture weeds.\n"
        )),
        HumanMessage(content=(
            f"Question: {question}\n\n"
            f"Gathered facts:\n{facts_str}"
        )),
    ])
    print(f"[Evaluation]: {result.status} | {result.reasoning}")
    return{
        "evaluation" : result.status,
        "missing_info" : result.missing_piece
    }
            

In [ ]:
class Reformulation(BaseModel):
    new_query: str = Field(description = "A specific search query to find the missing information.")
    rationale: str = Field(description = "Why this query should fill the gap.")

def reformulate(state: WeedState):
    question = state["original_question"]
    gathered = state.get("gathered_facts", [])
    facts_str = "\n".join(f" - {f}" for f in gathered)

    missing = state["missing_info"]

    reform_llm = llm.with_structured_output(Reformulation)
    result = reform_llm.invoke([
        SystemMessage(content=(
            "You are a query reformulation specialist. Given the original question, "
            "the facts gathered so far, and the identified information gap, "
            "create a NEW search query that targets the missing piece.\n\n"
            "Rules:\n"
            "- The query must be DIFFERENT from what would have produced the existing facts.\n"
            "- Be specific — use names, entities, or details from gathered facts.\n"
            "- Keep the query short (under 10 words)"
        )),
        HumanMessage(content=(
            f"Original question: {question}\n\n"
            f"Gathered facts:\n{facts_str}\n\n"
            f"Information gap: {missing}"
        )),
    ])

    print( f"Follow up query: \"{result.new_query}\" ({result.rationale})")

    return{
    "current_query" : result.new_query,
    }

In [ ]:
def respond(state: WeedState):
    question = state["original_question"]
    gathered = state.get("gathered_facts", [])
    chat_history = state.get("chat_history", [])
    evaluation = state.get("evaluation", "")

    if evaluation == "irrelevant":
        response_text = (
            "I'm specialized in weed identification "
            "and management only. Please ask me about "
            "weeds, control methods, or plant characteristics!"
        )
        updated_history = chat_history + [
            {"role": "user", "content": question},
            {"role": "assistant", "content": response_text}
        ]
        return {
            "response": response_text,
            "chat_history": updated_history
        }
    facts_str = "\n".join(f"- {f}" for f in gathered)
    history_str = "\n".join([
        f"{msg['role']}: {msg['content']}"
        for msg in chat_history
    ]) if chat_history else "No previous conversation"
    
    response = llm.invoke([
        SystemMessage(content=(
            f"You are an expert agricultural assistant.\n\n"
            f"Previous Conversation:\n{history_str}\n\n"
            f"Answer using ONLY these verified facts:\n"
            f"{facts_str}\n\n"
            "Instructions:\n"
            "- For weed ID: mention species name & characteristics\n"
            "- For control: mention all methods with timing\n"
            "- Be clear and simple for farmers\n"
            "- Do NOT use information beyond the facts"
        )),
        HumanMessage(content=question)
    ])
    
    updated_history = chat_history + [
        {"role": "user", "content": question},
        {"role": "assistant", "content": response.content}
    ]
    
    return {
        "response": response.content,
        "chat_history": updated_history
    }

In [ ]:
def evaluate_router(state: WeedState) -> Literal["answerable", "needs_more" , "irrelevant" ]:
    return state["evaluation"]

In [ ]:
from langgraph.graph import START , END ,StateGraph

workflow = StateGraph(WeedState)

workflow.add_node("retrieve" , retrieve_facts)
workflow.add_node("evaluate" , evaluate)
workflow.add_node("reformulate" , reformulate)
workflow.add_node("respond" , respond)

workflow.add_edge(START , "retrieve")
workflow.add_edge("retrieve" ,"evaluate")
workflow.add_conditional_edges("evaluate" , evaluate_router , {
    "answerable" : "respond",
    "needs_more" : "reformulate",
    "irrelevant" : "respond"
})
workflow.add_edge("reformulate" , "retrieve")
workflow.add_edge("respond", END)

graph = workflow.compile()

In [ ]:
question = "I have a weed with purple flower heads and spine-tipped bracts. Which thistle is this?"

chat_history = []

initial_state = {
    "original_question" : question,
    "current_query" : question,
    "documents" : [],
    "gathered_facts" : [],
    "evaluation" : None,
    "missing_info" : None,
    "response" : "",
    "chat_history" : chat_history
}

final_state = graph.invoke(initial_state)
print("\nFinal Answer:")
print(final_state["response"])

chat_history = final_state["chat_history"]
